In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
from src.retrieve_documents import retrieve_documents
from src.searching_tool import fit_docs, search

from src.build_prompt import build_prompt

In [1]:
from langchain_ollama import ChatOllama

In [10]:
retrieve_llm = ChatOllama(model="batiia-gemma-mini:latest")

In [14]:
def llm(prompt):
    response = retrieve_llm.invoke(prompt)
    return response.content

In [16]:
print(llm("Hey, what's up?"))

Not much, just hanging out and ready to chat! 😊

How about you? What's up going on?


In [17]:
## making hallucination examples due to not context

question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Please tell me which course you are referring to!

I don't have access to your personal accounts or the specific course you are looking at.

**To find out if you can join, you will need to check:**

1. **The Course Platform:** Go back to the website or platform where you found the course (e.g., Coursera, Udemy, edX, a university website).
2. **Enrollment Page:** Look for the "Enroll Now," "Register," or "Sign Up" button.
3. **Terms and Conditions:** Check the course description for any prerequisites or deadlines.

If you can provide the **name of the course** or the **website** where you found it, I might be able to give you more specific guidance!


In [19]:
## to fix this we add some context and a new promp
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

Yes, you can join. However, if you want to receive a certificate, you need to submit your project while they are still accepting submissions.


In [ ]:
documents = retrieve_documents()

index = fit_docs(documents)

In [ ]:
## Creating a function to make a simple RAG (example does not work)

def rag(question):
    search_results = search(question, index)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

This is an example of the Arch used to generate a RAG

```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```